In [ ]:


from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from rag import load_pdf_text, parse_constitution, make_embeddings, faiss_from_documents, save_vectorstore, as_retriever, rag_query,make_openrouter_chat_model


In [45]:
load_dotenv(override=True)

True

In [46]:
data = load_pdf_text("data/1999.pdf")
print("PDF loaded successfully")

PDF loaded successfully


In [47]:

chunks_documents = parse_constitution(data)
print(f"Total structured chunks: {len(chunks_documents)}")

# Preview
print(chunks_documents[0].page_content[:500])
print(chunks_documents[42].metadata)

Total structured chunks: 832
Section 1 - Supremacy of the Constitution
{'chapter': 'CHAPTER IV', 'part': 'PART II', 'section': '42', 'title': 'Right to freedom from discrimination', 'source': '1999 Constitution'}


In [48]:
embeddings = make_embeddings()
print("Embeddings loaded")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2895.14it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings loaded


In [49]:
vectorstore = faiss_from_documents(chunks_documents, embeddings)
save_vectorstore(vectorstore, "faiss_index")

print("Vector store created and saved")
print(len(vectorstore.index_to_docstore_id))

Vector store created and saved
832


In [50]:
retriever = as_retriever(vectorstore)

In [51]:
response = rag_query(
    "Who is authorized to remove the auditor general for the federation?",
    retriever=retriever,
    llm=make_openrouter_chat_model(),
)
print(response)


The President of the Federation is authorized to remove the Auditor-General for the Federation, acting on an address supported by a two-thirds majority of the Senate, as stated in Section 87(1) of the 1999 Constitution.


In [52]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)